1. Setup - Download Packages/Load Data


In [ ]:
# Packages and setup
# scikit-surprise gives us the CF toolkit: KNNBasic (the neighborhood model),
# BaselineOnly (our benchmark), Dataset/Reader (to wrap a DataFrame), and
# accuracy (RMSE/MAE). It requires numpy < 2.0 — its compiled parts were built
# against NumPy 1.x and break under 2.x.

# Run ONCE if surprise isn't installed (uncomment the lines for your setup):
# %pip install "numpy<2.0"
# %pip install scikit-surprise

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from surprise import KNNBasic, BaselineOnly, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split


In [ ]:
#Load data
books = pd.read_csv("Books.csv")
ratings = pd.read_csv("Ratings.csv")

# book_id is the unique key we use throughout; title is only for display,
# so we build a book_id -> title lookup
title_of = dict(zip(books["book_id"], books["title"]))

print(f"Books: {books.shape}  |  Ratings: {ratings.shape}")
books.head()


Books: (9964, 16)  |  Ratings: (164728, 3)
Out[13]:
book_id
isbn
authors
original_publication_year
title
language_code
average_rating
ratings_count
text_reviews_count
ratings_1
ratings_2
ratings_3
ratings_4
ratings_5
image_url
small_image_url
0
1
439023483
Suzanne Collins
2008.0
The Hunger Games (The Hunger Games, #1)
eng
4.34
4942365
155254
66715
127936
560092
1481305
2706317
https://images.gr-assets.com/books/1447303603m...
https://images.gr-assets.com/books/1447303603s...
1
2
439554934
J.K. Rowling, Mary GrandPrÃ©
1997.0
Harry Potter and the Sorcerer's Stone (Harry P...
eng
4.44
4800065
75867
75504
101676
455024
1156318
3011543
https://images.gr-assets.com/books/1474154022m...
https://images.gr-assets.com/books/1474154022s...
2
3
316015849
Stephenie Meyer
2005.0
Twilight (Twilight, #1)
en-US
3.57
3916824
95009
456191
436802
793319
875073
1355439
https://images.gr-assets.com/books/1361039443m...
https://images.gr-assets.com/books/1361039443s...
3
4
61120081
Harper Lee
1960.0
To Kill 

2. Exploratory Data Analysis


In [ ]:
# EDA (A) — the ratings themselves

print("Rating scale:", ratings['rating'].min(), "to", ratings['rating'].max())
print("Mean rating:   %.2f" % ratings['rating'].mean())
print("Median rating:", ratings['rating'].median())
print("\nCount of each rating value:")
print(ratings['rating'].value_counts().sort_index())

# % that clear the "relevant" bar we'll use in Precision/Recall later
print("\n%% of ratings >= 4 (our 'relevant' threshold): %.0f%%"
      % (100 * (ratings['rating'] >= 4).mean()))

ratings['rating'].value_counts().sort_index().plot(kind='bar')
plt.title("Distribution of ratings"); plt.xlabel("rating"); plt.ylabel("count")
plt.show()


Rating scale: 1 to 5
Mean rating:   3.84
Median rating: 4.0

Count of each rating value:
rating
1     3716
2    11382
3    42530
4    56933
5    50167
Name: count, dtype: int64

% of ratings >= 4 (our 'relevant' threshold): 65%


In [ ]:
# EDA Cont. — users, books, and sparsity
n_users = ratings['user_id'].nunique()
n_books = ratings['book_id'].nunique()
print(f"Users: {n_users}  |  Books rated: {n_books}  |  Catalog size: {books['book_id'].nunique()}")

# How full is the user x book matrix?
density = len(ratings) / (n_users * n_books)
print(f"Matrix density: {100*density:.2f}%  (so {100*(1-density):.2f}% is empty)")

# Books in the catalog that NOBODY rated -> CF can never recommend them
never_rated = (~books['book_id'].isin(ratings['book_id'])).sum()
print(f"Books in catalog with zero ratings: {never_rated}")

rpu = ratings.groupby('user_id').size()   # ratings per user
rpb = ratings.groupby('book_id').size()   # ratings per book
print(f"\nRatings per USER:  min {rpu.min()}, median {rpu.median():.0f}, max {rpu.max()}")
print(f"Ratings per BOOK:  min {rpb.min()}, median {rpb.median():.0f}, max {rpb.max()}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
rpu.plot(kind='hist', bins=30, ax=ax[0], title='Ratings per user')
rpb.plot(kind='hist', bins=30, ax=ax[1], title='Ratings per book')
ax[0].set_xlabel('# ratings'); ax[1].set_xlabel('# ratings')
plt.tight_layout(); plt.show()


Users: 1192  |  Books rated: 9229  |  Catalog size: 9964
Matrix density: 1.50%  (so 98.50% is empty)
Books in catalog with zero ratings: 735

Ratings per USER:  min 100, median 129, max 200
Ratings per BOOK:  min 1, median 8, max 100


Key EDA Insights
#### The ratings
- Ratings use a 1–5 integer scale (no half stars), so the model's
Reader
scale must be (1, 5).
- Ratings skew high: mean 3.84, median 4, and 65% of all ratings are a 4 or 5.
Only about 9% are a 1 or 2.
- This skew is selection bias. Readers choose books they expect to enjoy and
rate the ones they finish, so harsh ratings stay rare.
Why it matters for modeling
- A high, concentrated mean hands the popularity/mean baseline a strong start.
A model that predicts "around 3.84, nudged per user and per book" is right
most of the time, so the baseline will be hard to beat.
- We define a book as "relevant" at a rating of 4.0 for Precision/Recall.
Since 65% of ratings clear that bar, those scores read high by construction.
#### The users and books
- The sample has 1,192 users and 9,229 rated books, out of a 9,964-book catalog.
- The user-by-book matrix is 1.5% full, so 98.5% of possible ratings are blank.
Every prediction fills an empty cell from a thin slice of known ratings.
Sparsity is the central condition of the problem.
- Users and books are uneven in opposite directions:
- Every user rated 100 to 200 books (median 129). The data was filtered to
active users, so this sample has no new-user cold-start problem, even though
the lecture lists it as a standard challenge.
- Books are lopsided: the median book has 8 ratings, a few blockbusters hit
the 100-rating cap, and 735 catalog books have zero ratings.
Why it matters for modeling
- A book with one or two ratings can look perfect to collaborative filtering on
one or two data points. This justifies a minimum-ratings filter on the
recommendation candidates.
- The 735 never-rated books cannot surface through collaborative filtering.
They cap catalog coverage and motivate a content or LLM layer later.


3. Build Surprise Dataset


In [ ]:
# Build the Surprise dataset
# Goodreads ratings are integers 1 to 5 (confirmed in EDA), so the scale is (1, 5).
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user_id", "book_id", "rating"]], reader)

# build_full_trainset() trains on every rating. Good for the illustrative Top-N
# in Part 1. We switch to a 90/10 split for honest evaluation in Part 2.
full_trainset = data.build_full_trainset()
print(f"Trainset: {full_trainset.n_users} users, "
      f"{full_trainset.n_items} books, {full_trainset.n_ratings} ratings")


Trainset: 1192 users, 9229 books, 164728 ratings


4. Modeling Part 1: User vs Item Based CF


In [ ]:
# Build the four models and fit them
ubcf_pearson = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True},  verbose=False)
ubcf_cosine  = KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": True},  verbose=False)
ibcf_pearson = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": False}, verbose=False)
ibcf_cosine  = KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": False}, verbose=False)

for m in [ubcf_pearson, ubcf_cosine, ibcf_pearson, ibcf_cosine]:
    m.fit(full_trainset)


In [ ]:
# --- Top-N recommendations (popularity-filtered) ---
# EDA showed many books have very few ratings, so CF could rank a book highly on
# almost no evidence. Restrict candidates to books with >= MIN_RATINGS ratings.
# MIN_RATINGS=20 keeps ~26% of rated books; lower cutoffs trade trust for coverage.
MIN_RATINGS = 20
counts = ratings["book_id"].value_counts()
popular_books = set(counts[counts >= MIN_RATINGS].index)

def top_n_for_user(model, user_id, top_n=5):
    seen = set(ratings.loc[ratings["user_id"] == user_id, "book_id"])
    scored = [(title_of[b], round(model.predict(user_id, b).est, 3))
              for b in books["book_id"]
              if b not in seen and b in popular_books]
    return sorted(scored, key=lambda x: -x[1])[:top_n]


In [ ]:
TEST_USER = 314
for label, model in [("UBCF Pearson", ubcf_pearson), ("UBCF Cosine", ubcf_cosine),
                     ("IBCF Pearson", ibcf_pearson), ("IBCF Cosine", ibcf_cosine)]:
    print(f"\n--- {label} Top-5 for user {TEST_USER} ---")
    for title, score in top_n_for_user(model, TEST_USER, top_n=5):
        print(f"  {score:.3f}  {title}")


--- UBCF Pearson Top-5 for user 314 ---
  5.000  Blankets
  5.000  Lifeguard
  4.841  Frog and Toad Together (Frog and Toad, #2)
  4.820  Oh, The Places You'll Go!
  4.818  Season of Mists (The Sandman #4)

--- UBCF Cosine Top-5 for user 314 ---
  5.000  The Lord of the Rings (The Lord of the Rings, #1-3)
  4.901  A Court of Mist and Fury (A Court of Thorns and Roses, #2)
  4.900  Outlander (Outlander, #1)
  4.900  The Cat in the Hat and Other Dr. Seuss Favorites
  4.801  The Name of the Wind (The Kingkiller Chronicle, #1)

--- IBCF Pearson Top-5 for user 314 ---
  4.500  The Raven Boys (The Raven Cycle, #1)
  4.398  Red Seas Under Red Skies (Gentleman Bastard, #2)
  4.300  Can You Keep a Secret?
  4.270  A Little Life
  4.217  Metamorphoses

--- IBCF Cosine Top-5 for user 314 ---
  4.400  Dry
  4.400  Winter (The Lunar Chronicles, #4)
  4.400  Mother Night
  4.300  Of Mice and Men
  4.300  Animal, Vegetable, Miracle: A Year of Food Life


### Modeling Part 1 Insights: User-Based vs Item-Based CF
- For the same user (314), the two methods produced non-overlapping top-5 lists.
User-based returned Blankets, Lifeguard, Frog and Toad, Oh The Places You'll Go,
and a Sandman volume. Item-based returned Dry, Winter, Mother Night, Of Mice and
Men, and Animal Vegetable Miracle.
- The lists diverge because each method reasons from a different starting point.
User-based finds people who rate like user 314 and echoes what they loved
("people like you"). Item-based finds books rated like the ones 314 already
liked ("because you liked X").
- User-based predictions ran higher (4.8 to 5.0, two at the ceiling) and leaned
eclectic and niche, showing the serendipity strength. Item-based predictions
ran lower and flatter (4.3 to 4.4) and leaned toward well-known titles.
- Two plausible lists, no obvious winner. Reading them cannot pick a model. This
is why Part 2 holds out test ratings and scores each model on RMSE,
Precision@10, and Recall@10.


5. Modeling Part 2: Evaluation


In [ ]:
# --- Precision and Recall @ K (ranking quality) ---
# A book is "relevant" if its TRUE rating >= threshold (4.0, matching our EDA).
# For each user we rank their predictions by estimated rating, then ask:
#   precision = of the top_n we'd show, how many were actually relevant?
#   recall    = of all their relevant books, how many made the top_n?
# Returns the mean of each across all users.
def precision_recall_at_k(predictions, top_n=10, threshold=4.0):
    user_data = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_data[uid].append((est, true_r))

    precisions, recalls = [], []
    for items in user_data.values():
        items.sort(key=lambda x: x[0], reverse=True)   # rank by predicted rating
        n_relevant = sum(1 for _, t in items if t >= threshold)
        n_hits     = sum(1 for _, t in items[:top_n] if t >= threshold)
        precisions.append(n_hits / top_n)
        if n_relevant > 0:                              # avoid divide-by-zero
            recalls.append(n_hits / n_relevant)
    return np.mean(precisions), np.mean(recalls)


Train/Test Split


In [ ]:
# --- Train/Test split (90/10) ---
# Hold out 10% so we score predictions on ratings the models never trained on.
# random_state fixes the split so results reproduce (the course uses 6604).
trainset, testset = train_test_split(data, test_size=0.1, random_state=6604)
print(f"Train: {trainset.n_ratings} ratings  |  Test: {len(testset)} ratings")


Train: 148255 ratings  |  Test: 16473 ratings


In [ ]:
# --- Model comparison on the held-out test set ---
# Now testing FOUR CF variants plus the baseline:
# UBCF with Pearson (handles different rating scales well)
# UBCF with Cosine (captures co-occurrence patterns)
# IBCF with Pearson (less common but worth comparing)
# IBCF with Cosine (the standard item-based choice)
models = {
    "Baseline":        BaselineOnly(verbose=False),
    "UBCF Pearson":    KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True},  verbose=False),
    "UBCF Cosine":     KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": True},  verbose=False),
    "IBCF Pearson":    KNNBasic(k=10, sim_options={"name": "pearson", "user_based": False}, verbose=False),
    "IBCF Cosine":     KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": False}, verbose=False),
}

results = []
for name, m in models.items():
    m.fit(trainset)
    preds = m.test(testset)
    p, r = precision_recall_at_k(preds, top_n=10, threshold=4.0)
    results.append({
        "Model":        name,
        "RMSE":         round(accuracy.rmse(preds, verbose=False), 4),
        "Precision@10": round(p, 4),
        "Recall@10":    round(r, 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


Model   RMSE  Precision@10  Recall@10
    Baseline 0.8423        0.6568     0.7912
UBCF Pearson 1.0371        0.6565     0.7899
 UBCF Cosine 1.0201        0.6493     0.7795
IBCF Pearson 0.9018        0.6372     0.7693
 IBCF Cosine 0.9173        0.6245     0.7526


In [ ]:
# Fit all four CF variants on full trainset for Top-N illustration
ubcf_pearson = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True},  verbose=False)
ubcf_cosine  = KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": True},  verbose=False)
ibcf_pearson = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": False}, verbose=False)
ibcf_cosine  = KNNBasic(k=10, sim_options={"name": "cosine",  "user_based": False}, verbose=False)

for m in [ubcf_pearson, ubcf_cosine, ibcf_pearson, ibcf_cosine]:
    m.fit(full_trainset)

# Compare all four for user 314
TEST_USER = 314
for label, model in [("UBCF Pearson", ubcf_pearson), ("UBCF Cosine", ubcf_cosine),
                     ("IBCF Pearson", ibcf_pearson), ("IBCF Cosine", ibcf_cosine)]:
    print(f"\n--- {label} Top-5 for user {TEST_USER} ---")
    for title, score in top_n_for_user(model, TEST_USER, top_n=5):
        print(f"  {score:.3f}  {title}")


--- UBCF Pearson Top-5 for user 314 ---
  5.000  Blankets
  5.000  Lifeguard
  4.841  Frog and Toad Together (Frog and Toad, #2)
  4.820  Oh, The Places You'll Go!
  4.818  Season of Mists (The Sandman #4)

--- UBCF Cosine Top-5 for user 314 ---
  5.000  The Lord of the Rings (The Lord of the Rings, #1-3)
  4.901  A Court of Mist and Fury (A Court of Thorns and Roses, #2)
  4.900  Outlander (Outlander, #1)
  4.900  The Cat in the Hat and Other Dr. Seuss Favorites
  4.801  The Name of the Wind (The Kingkiller Chronicle, #1)

--- IBCF Pearson Top-5 for user 314 ---
  4.500  The Raven Boys (The Raven Cycle, #1)
  4.398  Red Seas Under Red Skies (Gentleman Bastard, #2)
  4.300  Can You Keep a Secret?
  4.270  A Little Life
  4.217  Metamorphoses

--- IBCF Cosine Top-5 for user 314 ---
  4.400  Dry
  4.400  Winter (The Lunar Chronicles, #4)
  4.400  Mother Night
  4.300  Of Mice and Men
  4.300  Animal, Vegetable, Miracle: A Year of Food Life


6. Data Visualization


In [ ]:
# ============================================================
# 6. PRESENTATION PLOTS (run after Sections 1-5)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Brand palette (consistent across all slides) ──────────────
C1  = "#1B4F72"   # deep navy   – primary bars
C2  = "#2E86C1"   # mid blue    – secondary
C3  = "#AED6F1"   # pale blue   – light fill
ACC = "#E74C3C"   # red accent  – highlights / baseline
GOLD= "#F39C12"   # gold        – best model callout
BG  = "#F8FBFF"   # near-white background

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.spines.left": False,
    "axes.grid":        True,
    "grid.color":       "#D6E4F0",
    "grid.linewidth":   0.8,
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})

# ------------------------------------------------------------------
# PLOT 1 – Rating Distribution (polished bar chart)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
counts = ratings['rating'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values,
              color=[C3, C3, C2, C1, C1], edgecolor="white", linewidth=1.5,
              width=0.65, zorder=3)

# Annotate each bar
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 600,
            f"{val/1000:.1f}K",
            ha="center", va="bottom", fontsize=10, color=C1, fontweight="bold")

ax.axhline(counts.mean(), color=ACC, linestyle="--", linewidth=1.4,
           label=f"Mean count ({counts.mean()/1000:.1f}K)")
ax.set_xlabel("Star Rating", fontsize=12, labelpad=8)
ax.set_ylabel("Number of Ratings", fontsize=12, labelpad=8)
ax.set_title("Rating Distribution — Goodreads Sample",
             fontsize=14, fontweight="bold", pad=14, color=C1)
ax.set_xticks([1,2,3,4,5])
ax.set_xticklabels(["★1","★2","★3","★4","★5"], fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x/1000:.0f}K"))
ax.legend(frameon=False, fontsize=10)

# Callout annotation
ax.annotate("65% of ratings\nare 4 or 5 ★",
            xy=(4.5, counts[5]), xytext=(3.3, 48000),
            fontsize=9, color=ACC, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=ACC, lw=1.3))

plt.tight_layout()
plt.savefig("plot1_rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# PLOT 2 – Ratings per User vs. per Book (side-by-side histograms)
# ------------------------------------------------------------------
rpu = ratings.groupby('user_id').size()
rpb = ratings.groupby('book_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, series, title, color, xlabel in zip(
        axes,
        [rpu, rpb],
        ["Ratings per User", "Ratings per Book"],
        [C1, C2],
        ["# of ratings given by a user", "# of ratings a book received"]):
    ax.hist(series, bins=30, color=color, edgecolor="white",
            linewidth=0.8, zorder=3)
    ax.axvline(series.median(), color=ACC, linestyle="--", linewidth=1.5,
               label=f"Median = {series.median():.0f}")
    ax.set_title(title, fontsize=13, fontweight="bold", color=C1, pad=10)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=6)
    ax.set_ylabel("Frequency", fontsize=11, labelpad=6)
    ax.legend(frameon=False, fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0f}"))

fig.suptitle("Activity Distribution — Users Are Active, Books Are Skewed",
             fontsize=14, fontweight="bold", color=C1, y=1.02)
plt.tight_layout()
plt.savefig("plot2_activity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# PLOT 3 – Model Comparison Bar Chart (RMSE + Precision@10 + Recall@10)
# ------------------------------------------------------------------
# Make sure results_df exists from Section 5
model_names = results_df["Model"].tolist()
x = np.arange(len(model_names))
width = 0.26

fig, ax = plt.subplots(figsize=(11, 5))

b1 = ax.bar(x - width, results_df["RMSE"],        width, label="RMSE (lower=better)",
            color=[ACC if "Baseline" in n else C1 for n in model_names],
            edgecolor="white", linewidth=1.2, zorder=3)
b2 = ax.bar(x,          results_df["Precision@10"], width, label="Precision@10",
            color=[ACC if "Baseline" in n else C2 for n in model_names],
            edgecolor="white", linewidth=1.2, zorder=3)
b3 = ax.bar(x + width,  results_df["Recall@10"],    width, label="Recall@10",
            color=[ACC if "Baseline" in n else C3 for n in model_names],
            edgecolor="white", linewidth=1.2, zorder=3)

# Value labels
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f"{bar.get_height():.3f}",
                ha="center", va="bottom", fontsize=8, color="#444444")

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel("Score", fontsize=12, labelpad=8)
ax.set_title("Model Evaluation — RMSE, Precision@10, Recall@10",
             fontsize=14, fontweight="bold", color=C1, pad=14)
ax.legend(frameon=False, fontsize=10, loc="upper right")

# Highlight baseline with a bracket label
ax.annotate("Baseline\n(hard to beat)",
            xy=(0, results_df.loc[results_df.Model=="Baseline","RMSE"].values[0]),
            xytext=(-0.5, 1.05),
            fontsize=8.5, color=ACC, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=ACC, lw=1.2))

plt.tight_layout()
plt.savefig("plot3_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# PLOT 4 – Sparsity Heatmap (10x10 sample of user-book matrix)
# ------------------------------------------------------------------
np.random.seed(42)
sample_users = np.random.choice(ratings['user_id'].unique(), 15, replace=False)
sample_books = np.random.choice(ratings['book_id'].unique(), 20, replace=False)
matrix_sample = (ratings[ratings['user_id'].isin(sample_users) &
                          ratings['book_id'].isin(sample_books)]
                 .pivot_table(index='user_id', columns='book_id',
                              values='rating', fill_value=0)
                 .reindex(index=sample_users, columns=sample_books, fill_value=0))

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(matrix_sample.values, cmap="Blues", aspect="auto",
               vmin=0, vmax=5, interpolation="nearest")

# Annotate non-zero cells
for i in range(matrix_sample.shape[0]):
    for j in range(matrix_sample.shape[1]):
        val = matrix_sample.values[i, j]
        if val > 0:
            ax.text(j, i, str(int(val)), ha="center", va="center",
                    fontsize=8, color="white" if val >= 4 else C1, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Rating (0 = unrated)", fontsize=10)
ax.set_title(f"User × Book Matrix — 98.5% Empty  (sample 15×20 shown)",
             fontsize=13, fontweight="bold", color=C1, pad=12)
ax.set_xlabel("Books (sample)", fontsize=11)
ax.set_ylabel("Users (sample)", fontsize=11)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig("plot4_sparsity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# PLOT 5 – Top-10 Most-Rated Books (horizontal bar)
# ------------------------------------------------------------------
top_books = (ratings.merge(books[["book_id","title","authors"]], on="book_id")
             .groupby("title")
             .agg(n_ratings=("rating","count"), avg_rating=("rating","mean"))
             .nlargest(10, "n_ratings")
             .reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
colors = [GOLD if i == 0 else C2 if i < 3 else C3
          for i in range(len(top_books))]
bars = ax.barh(top_books["title"][::-1], top_books["n_ratings"][::-1],
               color=colors[::-1], edgecolor="white", linewidth=1, height=0.65,
               zorder=3)

for bar, avg in zip(bars, top_books["avg_rating"][::-1]):
    ax.text(bar.get_width() + 0.5,
            bar.get_y() + bar.get_height()/2,
            f"★ {avg:.2f}",
            va="center", fontsize=9, color=C1)

ax.set_xlabel("Number of Ratings", fontsize=12, labelpad=8)
ax.set_title("Top 10 Most-Rated Books in the Dataset",
             fontsize=14, fontweight="bold", color=C1, pad=14)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0f}"))

# Short labels to avoid clipping
ax.set_yticklabels([t[:45]+"…" if len(t)>45 else t
                    for t in top_books["title"][::-1]], fontsize=9)

plt.tight_layout()
plt.savefig("plot5_top_books.png", dpi=150, bbox_inches="tight")
plt.show()

print("✅ All 5 presentation plots saved.")


/tmp/ipykernel_22798/2493142673.py:207: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([t[:45]+"…" if len(t)>45 else t
✅ All 5 presentation plots saved.


In [ ]:
# ============================================================
# 7. LLM RE-RANKING LAYER
# ============================================================

import getpass
from google import genai
from pydantic import BaseModel, Field

# ── Step 1: Connect to Gemini (key stays in memory only) ─────
GEMINI_KEY = getpass.getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=GEMINI_KEY)
MODEL  = "gemini-2.5-flash-lite"

print("Connected to Gemini.")


Enter your Gemini API key: ··········
Connected to Gemini.


In [ ]:
# ── Step 2: Define the output schema ─────────────────────────
# The model fills in these exact fields for every book it returns.

class BookPick(BaseModel):
    title:  str = Field(description="Exact book title from the candidate list.")
    reason: str = Field(description="One sentence on why this book fits the user's stated preference.")
    rank:   int = Field(description="Final rank position, 1 = best fit.")


In [ ]:
# ── Step 3: The re-ranking function ──────────────────────────
# ── Re-ranking function ───────────────────────────────────────
# Step 1: Get Top-N candidates from best CF model (not Baseline)
# Step 2: Enrich with book metadata from Books.csv
# Step 3: Build catalog string for the prompt
# Step 4: Send to Gemini with system instruction + structured output schema
# Step 5: Return parsed results

def llm_rerank(user_id, preference, top_n=10):
    """
    user_id    : int — the user to recommend for
    preference : str — mood/genre stated by the user
    top_n      : int — how many CF candidates to pass to the LLM
    """

    # ── Step 1: Pick best CF model that can generate Top-N ────
    # Baseline wins RMSE but cannot generate ranked lists (no neighbors)
    # so we pick the best KNN model by RMSE instead
    knn_results = results_df[results_df["Model"] != "Baseline"].copy()
    best_name   = knn_results.loc[knn_results["RMSE"].idxmin(), "Model"]

    model_map = {
        "UBCF Pearson": ubcf_pearson,
        "UBCF Cosine":  ubcf_cosine,
        "IBCF Pearson": ibcf_pearson,
        "IBCF Cosine":  ibcf_cosine,
    }
    cf_model = model_map[best_name]
    print(f"CF model used for candidates: {best_name}")

    # ── Step 2: Get Top-N candidates from CF model ────────────
    cf_recs = top_n_for_user(cf_model, user_id, top_n=top_n)
    # cf_recs is a list of (title, predicted_score)

    # ── Step 3: Enrich with metadata from Books.csv ───────────
    meta = books.set_index("title")[
        ["authors", "original_publication_year", "average_rating"]
    ].to_dict("index")

    candidates = []
    for title, cf_score in cf_recs:
        info = meta.get(title, {})
        author     = str(info.get("authors", "Unknown"))[:40]
        year_raw   = info.get("original_publication_year", 0)
        year       = int(year_raw) if year_raw and not pd.isna(year_raw) else 0
        avg_raw    = info.get("average_rating", 0)
        avg_rating = round(float(avg_raw), 2) if avg_raw and not pd.isna(avg_raw) else 0.0
        candidates.append({
            "title":      title,
            "author":     author,
            "year":       year,
            "avg_rating": avg_rating,
            "cf_score":   cf_score,
        })

    # ── Step 4: Build catalog string for the prompt ───────────
    # Mirrors class demo: catalog = "\n".join(f"- {m['title']} [{m['genres']}]" ...)
    catalog = "\n".join(
        f"- {c['title']} by {c['author']} "
        f"({c['year']}) | avg rating: {c['avg_rating']} | CF score: {c['cf_score']}"
        for c in candidates
    )

    # ── Step 5: System instruction ────────────────────────────
    # Mirrors the concierge pattern from the class demo
    system_instruction = (
        "You are a personal book concierge. "
        "You will receive a list of book candidates selected by a collaborative "
        "filtering algorithm for a specific user. "
        "Re-rank these candidates based on how well they match the user's "
        "stated preference. "
        "Use ONLY the books in the provided list — do not suggest new books. "
        "Return every book from the list in your re-ranked order. "
        "Give each book a one-sentence reason tied specifically to the "
        "user's stated preference."
    )

    # ── Step 6: Call Gemini with structured output ────────────
    # Exactly mirrors the class pattern:
    #   response_mime_type  = "application/json"
    #   response_schema     = list[BookPick]
    #   response.parsed     = ready-to-use Python objects
    response = client.models.generate_content(
        model=MODEL,
        contents=(
            f"User's preference: {preference}\n\n"
            f"Book candidates from collaborative filtering:\n{catalog}"
        ),
        config={
            "system_instruction": system_instruction,
            "response_mime_type": "application/json",
            "response_schema":    list[BookPick],
            "temperature":        0.7,
            "max_output_tokens":  1000,
        },
    )

    return response.parsed, candidates


In [ ]:
# ── Run for test user ─────────────────────────────────────────
TEST_USER  = 314
PREFERENCE = "I want something dark, thought-provoking, and literary"

print(f"User {TEST_USER} says: \"{PREFERENCE}\"")
print("─" * 55)

reranked, cf_candidates = llm_rerank(
    user_id    = TEST_USER,
    preference = PREFERENCE,
    top_n      = 10,
)

# Print results
print(f"\n{'RANK':<5} {'TITLE':<45} REASON")
print("─" * 110)
for pick in sorted(reranked, key=lambda x: x.rank):
    title_short = pick.title[:43] + "…" if len(pick.title) > 43 else pick.title
    print(f"#{pick.rank:<4} {title_short:<45} {pick.reason}")


User 314 says: "I want something dark, thought-provoking, and literary"
───────────────────────────────────────────────────────
CF model used for candidates: IBCF Pearson

RANK  TITLE                                         REASON
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
#1    A Little Life by Hanya Yanagihara (2015)      This novel is known for its intense emotional depth and exploration of difficult themes, aligning with your preference for dark and thought-provoking literature.
#2    Metamorphoses by Ovid, David Raeburn, Denis…  This classical work offers profound mythological narratives that are inherently thought-provoking and literary, with darker undertones present in many of the transformations.
#3    Rosencrantz and Guildenstern Are Dead by To…  This play is a highly literary and philosophical work that delves into existential themes, fitting your request for thought-provoking content.
#4    The Raven Boys (

In [ ]:
# ── Before vs After — the key slide exhibit ──────────────────
# Shows CF order vs LLM re-ranked order side by side

print("=" * 70)
print(f"PREFERENCE: \"{PREFERENCE}\"")
print("=" * 70)
print(f"{'CF RANK':<9} {'TITLE':<40} {'LLM RANK':<10} MOVED")
print("─" * 70)

llm_rank_lookup = {p.title: p.rank for p in reranked}

for i, c in enumerate(cf_candidates, 1):
    title      = c["title"]
    new_rank   = llm_rank_lookup.get(title, i)
    title_s    = title[:38] + "…" if len(title) > 38 else title
    diff       = i - new_rank
    if diff > 0:
        moved = f"↑ +{diff}"
    elif diff < 0:
        moved = f"↓ {diff}"
    else:
        moved = "  ="
    print(f"CF #{i:<5}  {title_s:<40} LLM #{new_rank:<4}  {moved}")

print("\n── Top 3 picks with reasons ──")
for pick in sorted(reranked, key=lambda x: x.rank)[:3]:
    print(f"\n#{pick.rank}  {pick.title}")
    print(f"    → {pick.reason}")


PREFERENCE: "I want something dark, thought-provoking, and literary"
CF RANK   TITLE                                    LLM RANK   MOVED
──────────────────────────────────────────────────────────────────────
CF #1      The Raven Boys (The Raven Cycle, #1)     LLM #1       =
CF #2      Red Seas Under Red Skies (Gentleman Ba…  LLM #2       =
CF #3      Can You Keep a Secret?                   LLM #3       =
CF #4      A Little Life                            LLM #4       =
CF #5      Metamorphoses                            LLM #5       =
CF #6      Rosencrantz and Guildenstern Are Dead    LLM #6       =
CF #7      Bared to You (Crossfire, #1)             LLM #7       =
CF #8      Something Blue (Darcy & Rachel, #2)      LLM #8       =
CF #9      Where She Went (If I Stay, #2)           LLM #9       =
CF #10     The Constant Princess (The Plantagenet…  LLM #10      =

── Top 3 picks with reasons ──

#1  A Little Life by Hanya Yanagihara (2015)
    → This novel is known for its intense em